<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-14/NB14_rl_pinn_digital_twins.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 14: Learning to Act, Respecting Physics and Engineering Responsibly

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook teaches the heating agent of Week 1 by Q-learning and compares the learned policy with the hand-written hysteresis rule. It balances the cart-pole of Gymnasium with a linear policy found by the cross-entropy method [1, 2]. A physics-informed network then solves a damped oscillator and is compared with a data-only network [3], and a small digital twin identifies the room's thermal parameters from noisy data and forecasts the temperature [4]. The notebook ends with a responsible-AI checklist for the final project [5, 6].

This notebook is the hands-on part of Week 14. It opens with Python step 14: Environments, learning and organised code and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-14/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-14/Week14_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-14/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-14/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 14 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

The last lines of the setup cell add a hint to every `NameError`. This error appears when a cell uses a name that no cell has defined in the current session, for example because the cells above have not run.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=False)
    import gymnasium as gym

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
torch.manual_seed(0)

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

# Hint for a NameError: The usual cause in a notebook is a cell above that has not run in this session.
from IPython import get_ipython


def name_error_hint(shell, etype, value, tb, tb_offset=None):
    """Show a NameError as usual and add the hint."""
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("Hint: A name defined in a cell above exists only after that cell has run in this session. "
          "Runtime > Run before (Ctrl+F8) runs all cells above the current one.")


if get_ipython() is not None:
    get_ipython().set_custom_exc((NameError,), name_error_hint)

## Python step 14: Environments, learning and organised code

> **Starting here?** Run the setup cell above first. The first code cell below checks this and stops with a message if the setup has not run in this session.

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Functions with clear contracts

Programs grow, and the code of a final project is read by others and by its author months later. Small functions with descriptive names, type hints and docstrings make that reading easy. Type hints such as `temp_C: float` state what a parameter should be and what the function returns. Python does not enforce them, but editors and readers rely on them. The room model of Week 1 becomes a function with default parameters.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell above has not run in this session. "
                       "Run it first, then continue with this step.")

In [ ]:
def room_step(temp_C: float, outside_C: float, heater: int,
              tau_min: float = 240.0, rate_C_per_min: float = 0.12, dt_min: float = 5.0) -> float:
    """Room temperature after one time step of the first-order model of Week 1."""
    return temp_C + (outside_C - temp_C) * dt_min / tau_min + rate_C_per_min * dt_min * heater


print(room_step(20.0, 0.0, 1))

### Parameters in one place

A dataclass is a class that mainly stores values. The decorator `@dataclass` writes the `__init__` method automatically from the annotated fields, and `replace` creates a modified copy. Keeping all parameters of a model in one object avoids numbers scattered through the code and documents each variant of an experiment.

In [ ]:
from dataclasses import dataclass, replace


@dataclass
class RoomParams:
    tau_min: float = 240.0
    rate_C_per_min: float = 0.12
    dt_min: float = 5.0


base = RoomParams()
insulated = replace(base, tau_min=400.0)
print(base)
print(insulated.tau_min, base.tau_min)

### A shared interface

Gymnasium environments share one interface: `reset` returns the first observation, and `step` returns the next observation, the reward, two flags for the end of an episode and a dictionary of extra information [1]. A simulation written with the same interface works with any agent that works with Gymnasium. The class below wraps the heating task of this week.

In [ ]:
class RoomEnv:
    """Heating task with the reset and step interface of Gymnasium environments."""

    def __init__(self, params: RoomParams, outside_C: float = -3.5, target_C: float = 21.0, horizon: int = 288):
        self.p, self.outside, self.target, self.horizon = params, outside_C, target_C, horizon

    def reset(self, temp_C: float = 18.0):
        self.temp, self.k = temp_C, 0
        return self.temp, {}

    def step(self, action: int):
        p = self.p
        self.temp = room_step(self.temp, self.outside, action, p.tau_min, p.rate_C_per_min, p.dt_min)
        self.k += 1
        reward = -abs(self.temp - self.target)
        return self.temp, reward, False, self.k >= self.horizon, {}


env = RoomEnv(base)
obs, info = env.reset()
total = 0.0
for _ in range(12):
    obs, reward, terminated, truncated, info = env.step(1 if obs < 21 else 0)
    total += reward
print(f"temperature {obs:.2f} °C, return {total:.2f}")

### Learning to act with tabular Q-learning

Q-learning estimates the value Q(s, a) of taking action a in state s and acting well afterwards [7]. After every step the estimate moves towards the observed reward plus the discounted value of the best action in the next state: Q(s, a) += lr (r + gamma max Q(s', .) - Q(s, a)). A table needs discrete states, so `np.digitize` maps the temperature to one of seven bands. The agent explores with a small probability epsilon and otherwise takes the action with the highest estimate. The environment class above serves unchanged.

In [ ]:
import numpy as np

bands = np.array([19.0, 20.0, 20.5, 21.0, 21.5, 22.0])     # band edges in °C: seven states


def state_of(temp_C: float) -> int:
    return int(np.digitize(temp_C, bands))


Q = np.zeros((len(bands) + 1, 2))                          # one row per state, one column per action
agent_rng = np.random.default_rng(14)
lr, gamma, eps = 0.1, 0.95, 0.1
env_q = RoomEnv(base, horizon=96)                          # episodes of eight hours
for episode in range(300):
    obs, _ = env_q.reset(temp_C=float(agent_rng.uniform(17.0, 23.0)))
    s, done = state_of(obs), False
    while not done:
        a = int(agent_rng.integers(2)) if agent_rng.random() < eps else int(Q[s].argmax())
        obs, r, terminated, truncated, _ = env_q.step(a)
        s2 = state_of(obs)
        Q[s, a] += lr * (r + gamma * Q[s2].max() - Q[s, a])
        s, done = s2, terminated or truncated
print(np.round(Q, 1))
print("heater on in states:", np.flatnonzero(Q.argmax(axis=1) == 1))

The learned policy switches the heater on in the cold bands and off in the warm ones, a thermostat rule discovered from rewards alone. The deep reinforcement learning of this week replaces the table by a neural network when the state has many continuous variables.

### Physics in the loss

Physics-informed learning adds the residual of a governing equation to the loss of a model. The room obeys dT/dt = (T_out - T) / tau + r u. For a candidate temperature curve T(t), automatic differentiation provides dT/dt at many time points, and the mean squared residual measures how strongly the curve violates the physics. Summing T before differentiating works because every value of T depends only on its own time point. The exact solution of the unheated room gives a residual of practically zero, and a straight line does not; a physics-informed network minimises such a residual together with the error on measured data.

In [ ]:
import torch

t = torch.linspace(0.0, 240.0, 25, requires_grad=True)     # minutes
T_out, T0, tau = -3.5, 18.0, 240.0


def physics_loss(T, u=0.0, rate=0.12):
    """Mean squared residual of dT/dt = (T_out - T) / tau + rate u at the points t."""
    dTdt, = torch.autograd.grad(T.sum(), t, create_graph=True)
    return ((dTdt - (T_out - T) / tau - rate * u) ** 2).mean()


exact = T_out + (T0 - T_out) * torch.exp(-t / tau)
line = T0 - 0.05 * t
print(f"exact solution: {physics_loss(exact).item():.1e}, straight line: {physics_loss(line).item():.1e}")

### Tests

A test is a small piece of code that checks an expected property. The statement `assert condition` does nothing when the condition is true and raises an `AssertionError` otherwise. Physical expectations make good tests: With no temperature difference and the heater off, the room must not change, and heating must raise the temperature. Running such tests after every change catches mistakes early.

In [ ]:
def test_room_step():
    assert abs(room_step(15.0, 15.0, 0) - 15.0) < 1e-12        # no difference, no heating
    assert room_step(15.0, 15.0, 1) > 15.0                     # heating raises the temperature
    assert abs(room_step(20.0, 0.0, 0) - (20.0 - 20.0 * 5 / 240)) < 1e-12
    print("all tests passed")


test_room_step()

### Modules and records of runs

Functions used in several notebooks belong in a module, a `.py` file that `import` loads. In a notebook, the first line `%%writefile` saves the rest of a cell as a file. Results should be stored together with the information needed to reproduce them: The versions of Python and of the main libraries, the parameters and the random seeds. `json` writes such a record in a readable form.

In [ ]:
%%writefile roomtools.py
"""Helpers for the heating examples of the course."""


def room_step(temp_C, outside_C, heater, tau_min=240.0, rate_C_per_min=0.12, dt_min=5.0):
    """Room temperature after one time step of the first-order model."""
    return temp_C + (outside_C - temp_C) * dt_min / tau_min + rate_C_per_min * dt_min * heater

In [ ]:
import json
import sys
import numpy as np
import roomtools

temps = [18.0]
for _ in range(288):
    temps.append(roomtools.room_step(temps[-1], -3.5, 1 if temps[-1] < 21 else 0))
record = {"python": sys.version.split()[0], "numpy": np.__version__,
          "params": vars(base), "final_temp_C": round(temps[-1], 3), "steps": len(temps) - 1}
with open("run_record.json", "w", encoding="utf-8") as f:
    json.dump(record, f, indent=1)
print(json.dumps(record))

A final project can follow the same habits with a simple layout: A README that states the problem and how to run the code, a script or notebook that obtains the data, notebooks for exploration, a module for shared functions, and a results folder with figures and run records.

**Quick check.** What happens when the line assert room_step(15.0, 15.0, 1) < 15.0 runs?

A. It prints False and continues  
B. It raises an AssertionError, because heating raises the temperature  
C. Nothing, because assert statements are only comments  
D. It returns 15.0

<details><summary>Show the answer</summary>

**B.** The condition is false, so assert stops with an AssertionError, which is how a failing test announces itself.

</details>

### Exercises for Python step 14

**Exercise 14.1.** Create a `RoomEnv` with the parameters `insulated`, reset it, run 24 steps with the rule "heat when the temperature is below 21 °C", and store the final temperature in `py_final_insulated`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py14_final_insulated_reference():
    t = 18.0
    for _ in range(24):
        a = 1 if t < 21 else 0
        t = t + (-3.5 - t) * 5.0 / 400.0 + 0.12 * 5.0 * a
    return t

In [ ]:
py_final_insulated = None
check("Exercise 14.1", py_final_insulated, _py14_final_insulated_reference())

**Exercise 14.2.** Write a function `energy_used(actions, power_kW=2.0, dt_min=5.0)` with type hints and a docstring that returns the heating energy in kWh for a list of heater commands. Store `energy_used([1, 1, 0, 1])` in `py_energy`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py14_energy_reference():
    return sum([1, 1, 0, 1]) * 2.0 * 5.0 / 60.0

In [ ]:
py_energy = None
check("Exercise 14.2", py_energy, _py14_energy_reference())

**Exercise 14.3.** Write a test function `test_energy_used` with at least two `assert` statements, for example that no heating uses no energy and that twelve steps of heating use 2 kWh. Run it. This exercise has no automatic check, because a passing test prints nothing unless it is asked to.

In [ ]:
# write test_energy_used here and call it

**Exercise 14.4.** Use `state_of` and the learned table `Q` to find the action that the greedy policy takes at 19.5 °C, and store it (0 or 1) in `py_action_cold`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py14_action_cold_reference():
    return 1

In [ ]:
py_action_cold = None
check("Exercise 14.4", py_action_cold, _py14_action_cold_reference())

**Exercise 14.5.** With the heater on, the room settles at T_eq = T_out + r tau. Write the exact solution for u = 1 by analogy with `exact`, compute its physics loss with `physics_loss(..., u=1.0)`, and store the number in `py_heated_loss`. A value close to zero confirms the solution.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py14_heated_loss_reference():
    return 0.0

In [ ]:
py_heated_loss = None
check("Exercise 14.5", py_heated_loss, _py14_heated_loss_reference())

## 1. One Q-learning update

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

### Your turn, exercise 1

With Q(s, a) = 2.0, reward r = -1, discount gamma = 0.9, best next value max Q(s', a') = 3.0 and learning rate alpha = 0.5, compute the updated Q(s, a) and store it in `q_new` [8].

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _q_new_reference():
    return 2.0 + 0.5 * (-1 + 0.9 * 3.0 - 2.0)

In [ ]:
q_new = None   # your computation
check("Exercise 1", q_new, _q_new_reference())

## 2. The thermostat learns its own rule

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

The room model of Week 1 becomes an environment. The state is the room temperature in bins of 0.5 C together with the previous heater state; the actions are heater off and on. The reward penalises the distance from 21 C, every switching action and, a little, every step of heating. The agent trains on 400 synthetic winter days with an epsilon-greedy policy [7].

**Python move: a Q-table as a NumPy array.** `Q[s, u, a]` holds the value of action `a` in temperature bin `s` with previous heater state `u`.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
BINS = np.arange(15, 27.01, 0.5)
bin_of = lambda T: int(np.clip(np.round((T - 15) / 0.5), 0, len(BINS) - 1))
step_env = lambda T, u, out: T + (out - T) * 5 / 240 + 0.12 * 5 * u


def winter_day(seed):
    r = np.random.default_rng(seed)
    base = r.uniform(-3, 4)
    return base - 5 * np.cos(2 * np.pi * (np.arange(288) / 12 - 4) / 24) + 0.5 * r.normal(size=288)


def train_q(days=400, eps=0.1, alpha=0.2, gamma=0.95, w_switch=0.3, w_energy=0.05, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((len(BINS), 2, 2))
    for d in range(days):
        out, T, u = winter_day(1000 + d), rng.uniform(17, 24), 0
        for k in range(288):
            s = bin_of(T)
            a = int(rng.integers(2)) if rng.random() < eps else int(np.argmax(Q[s, u]))
            T2 = step_env(T, a, out[k])
            reward = -abs(T2 - 21) - w_switch * (a != u) - w_energy * a
            Q[s, u, a] += alpha * (reward + gamma * Q[bin_of(T2), a].max() - Q[s, u, a])
            T, u = T2, a
    return Q


def run_policy(policy, out):
    T, u, switches, err, trace = 19.0, 0, 0, 0.0, []
    for k in range(288):
        a = policy(T, u)
        switches += a != u
        u = a
        T = step_env(T, u, out[k])
        trace.append(T); err += abs(T - 21)
    return np.array(trace), switches, err / 288


Q = train_q()
learned = lambda T, u: int(np.argmax(Q[bin_of(T), u]))
hysteresis = lambda T, u: 1 if T < 20.5 else (0 if T > 21.5 else u)
test_day = winter_day(7)
hours = np.arange(288) / 12
plt.figure(figsize=(8, 3.3))
for name, pol in [("learned by Q-learning", learned), ("hysteresis, 1 C band", hysteresis)]:
    tr, sw, e = run_policy(pol, test_day)
    print(f"{name:22s} mean |error| {e:.2f} C, switches {sw}")
    plt.plot(hours, tr, label=name)
plt.axhline(21, color="gray", ls="--", lw=0.8); plt.xlabel("hour"); plt.ylabel("room (C)"); plt.legend(); plt.show()
policy_table = np.array([[learned(T, u) for T in BINS] for u in (0, 1)])
print("heater command by temperature bin from 15 to 27 C (rows: heater was off / on):")
print("\n".join(" ".join(str(v) for v in row) for row in policy_table))

Read the table from left to right: At low temperatures the learned command is 1, at high temperatures 0, and the switch points differ between the two rows, so the agent has discovered a hysteresis band by itself. Entries at temperatures that the room rarely reaches can look erratic, because the agent has little experience there; this is a general property of learned policies outside the conditions seen in training.

### Your turn, exercise 2

Retrain with a switching penalty of 2.0 instead of 0.3 (all other settings unchanged) and store the number of switches of the learned policy on `test_day` in `switches_high_penalty`. How did the reward design change the behaviour?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _switches_high_penalty_reference():
    Qh = train_q(w_switch=2.0)
    return int(run_policy(lambda T, u: int(np.argmax(Qh[bin_of(T), u])), test_day)[1])

In [ ]:
switches_high_penalty = None   # your computation
check("Exercise 2", switches_high_penalty, _switches_high_penalty_reference())

## 3. CartPole with the cross-entropy method

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

The cart-pole task asks an agent to balance a pole on a moving cart [2]. A linear policy pushes right when w . observation + b > 0. The cross-entropy method samples many weight vectors, keeps the best fifth and refits the sampling distribution to them. It is simple and needs no gradients, like the population methods of Weeks 5 and 6 [1].

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
env = gym.make("CartPole-v1")


def episode_return(params, seed):
    obs, _ = env.reset(seed=int(seed))
    total = 0.0
    for _ in range(500):
        obs, reward, terminated, truncated, _ = env.step(int(obs @ params[:4] + params[4] > 0))
        total += reward
        if terminated or truncated:
            break
    return total


rng = np.random.default_rng(0)
mean, std = np.zeros(5), np.ones(5)
for it in range(12):
    samples = mean + std * rng.normal(size=(40, 5))
    scores = np.array([episode_return(p, 1000 * it + j) for j, p in enumerate(samples)])
    elite = samples[np.argsort(scores)[-8:]]
    mean, std = elite.mean(0), elite.std(0) + 0.05
    print(f"iteration {it + 1:2d}: mean return {scores.mean():6.1f}, best {scores.max():5.0f}")
print("average over 10 test episodes:", np.mean([episode_return(mean, 99999 - k) for k in range(10)]))

## 4. A physics-informed network for a damped oscillator

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

The oscillator obeys x'' + 0.4 x' + 4 x = 0 with x(0) = 1 and x'(0) = 0, whose exact solution is known. A data-only network sees eight noisy measurements from the first 1.5 seconds. The physics-informed network sees the same data plus the equation residual at 200 collocation points over 0 to 6 seconds [3, 9].

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def exact(t, zeta=0.1, w=2.0):
    wd = w * math.sqrt(1 - zeta ** 2)
    return np.exp(-zeta * w * t) * (np.cos(wd * t) + zeta * w / wd * np.sin(wd * t))


r = np.random.default_rng(1)
t_data = np.sort(r.uniform(0, 1.5, 8)); x_data = exact(t_data) + 0.02 * r.normal(size=8)
td = torch.tensor(t_data, dtype=torch.float32)[:, None]; xd = torch.tensor(x_data, dtype=torch.float32)[:, None]
tc = torch.linspace(0, 6, 200)[:, None].requires_grad_(True)
t0 = torch.zeros(1, 1, requires_grad=True)


def make_net():
    return nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))


def derivatives(net, t):
    x = net(t)
    dx = torch.autograd.grad(x.sum(), t, create_graph=True)[0]
    d2x = torch.autograd.grad(dx.sum(), t, create_graph=True)[0]
    return x, dx, d2x


def train(physics, steps=2500):
    torch.manual_seed(0)
    net = make_net(); opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(steps):
        opt.zero_grad()
        loss = ((net(td) - xd) ** 2).mean()
        if physics:
            x, dx, d2x = derivatives(net, tc)
            x0, dx0, _ = derivatives(net, t0)
            loss = loss + ((d2x + 0.4 * dx + 4 * x) ** 2).mean() + (x0 - 1).pow(2).mean() + dx0.pow(2).mean()
        loss.backward(); opt.step()
    return net


data_net, pinn = train(False), train(True)
tt = np.linspace(0, 6, 400)
with torch.no_grad():
    y_data = data_net(torch.tensor(tt, dtype=torch.float32)[:, None]).numpy().ravel()
    y_pinn = pinn(torch.tensor(tt, dtype=torch.float32)[:, None]).numpy().ravel()
plt.figure(figsize=(8, 3.3))
plt.plot(tt, exact(tt), "k", lw=2, label="exact"); plt.plot(tt, y_data, label="data-only network"); plt.plot(tt, y_pinn, "--", label="physics-informed network")
plt.scatter(t_data, x_data, color="C3", zorder=3, label="measurements"); plt.ylim(-1.5, 1.5); plt.xlabel("t (s)"); plt.ylabel("x"); plt.legend(fontsize=8); plt.show()
print(f"RMSE over 0 to 6 s: data-only {np.sqrt(np.mean((y_data - exact(tt)) ** 2)):.3f}, physics-informed {np.sqrt(np.mean((y_pinn - exact(tt)) ** 2)):.3f}")

## 5. A small digital twin of the room

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

Two days of measured room and outside temperatures and heater states are available, with sensor noise. Each five-minute change obeys dT = a (T_out - T) + b u, with a = dt / tau and b = heat rate times dt. Least squares estimates a and b; the identified model is the core of a twin that forecasts the room and can test controllers before they touch the real building [4, 10].

**Python move: np.linalg.lstsq.** It solves an overdetermined linear system in the least-squares sense.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "hysteresis", "step_env", "winter_day"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
true_tau, true_rate = 240.0, 0.12
rng = np.random.default_rng(5)
out = np.concatenate([winter_day(50), winter_day(51)])
T, u, Ts, Us = 19.0, 0, [], []
for k in range(len(out)):
    u = hysteresis(T, u)
    Ts.append(T); Us.append(u)
    T = step_env(T, u, out[k])
Ts, Us = np.array(Ts), np.array(Us)
meas = Ts + 0.05 * rng.normal(size=len(Ts))
dT = np.diff(meas)
A = np.column_stack([out[:-1] - meas[:-1], Us[:-1]])
(a, b), *_ = np.linalg.lstsq(A, dT, rcond=None)
print(f"identified tau = {5 / a:.1f} min (true {true_tau}), heat rate = {b / 5:.4f} C/min (true {true_rate})")

### Your turn, exercise 3

Use the identified parameters to simulate the room forward for the first six hours (72 steps) of the second day from the measured starting temperature, with the recorded heater states, and store the mean absolute difference between the twin's trajectory and the measurements in `twin_mae`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _twin_mae_reference():
    start = 288
    T, sim = meas[start], []
    for k in range(start, start + 72):
        sim.append(T)
        T = T + a * (out[k] - T) + b * Us[k]
    return float(np.mean(np.abs(np.array(sim) - meas[start:start + 72])))

In [ ]:
twin_mae = None   # your computation
check("Exercise 3", twin_mae, _twin_mae_reference(), tol=1e-6)

## 6. A responsible-AI checklist for the final project

The dictionary below maps each function of the NIST AI RMF to questions for the final project [6]. Answer them in the project report, and state which category of the EU AI Act would apply if the system were placed on the market [5].

In [ ]:
CHECKLIST = {
    "Govern": ["Who is accountable for the system and its decisions?", "Which rules govern data, access and changes?"],
    "Map": ["What is the intended use, and who is affected?", "Under which conditions could the system fail or be misused?"],
    "Measure": ["Which metrics, baselines and test conditions show that it works?", "How are errors tracked after deployment?"],
    "Manage": ["What happens when a risk exceeds tolerance: fallback, human override, retirement?", "How are users informed about limits?"],
}
for function, questions in CHECKLIST.items():
    print(function)
    for q in questions:
        print("  -", q)

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-14/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-14/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-14/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-14/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-14/README.md#research-and-report-assignment-optional) are listed in the week overview, and the [final project](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/exams/final/README.md) closes the course.

## References

[1] Towers, M., Kwiatkowski, A., Terry, J., et al. (2024). Gymnasium: A standard interface for reinforcement learning environments. arXiv preprint arXiv:2407.17032. <https://arxiv.org/abs/2407.17032>

[2] Barto, A. G., Sutton, R. S., & Anderson, C. W. (1983). Neuronlike adaptive elements that can solve difficult learning control problems. *IEEE Transactions on Systems, Man, and Cybernetics*, *SMC-13*(5), 834-846. <https://doi.org/10.1109/TSMC.1983.6313077>

[3] Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics*, *378*, 686-707. <https://doi.org/10.1016/j.jcp.2018.10.045>

[4] Grieves, M., & Vickers, J. (2017). Digital twin: Mitigating unpredictable, undesirable emergent behavior in complex systems. In *Transdisciplinary Perspectives on Complex Systems (F.-J. Kahlen, S. Flumerfelt, & A. Alves, Eds.)* (pp. 85-113). Springer. <https://doi.org/10.1007/978-3-319-38756-7_4>

[5] European Parliament and Council of the European Union (2024). *Regulation (EU) 2024/1689 laying down harmonised rules on artificial intelligence (Artificial Intelligence Act)*. Official Journal of the European Union, L series, 12 July 2024. <https://eur-lex.europa.eu/eli/reg/2024/1689/oj>

[6] National Institute of Standards and Technology (2023). *Artificial Intelligence Risk Management Framework (AI RMF 1.0), NIST AI 100-1*. NIST. <https://doi.org/10.6028/NIST.AI.100-1>

[7] Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

[8] Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning*, *8*(3-4), 279-292. <https://doi.org/10.1007/BF00992698>

[9] Karniadakis, G. E., Kevrekidis, I. G., Lu, L., Perdikaris, P., Wang, S., & Yang, L. (2021). Physics-informed machine learning. *Nature Reviews Physics*, *3*(6), 422-440. <https://doi.org/10.1038/s42254-021-00314-5>

[10] Tao, F., Zhang, H., Liu, A., & Nee, A. Y. C. (2019). Digital twin in industry: State-of-the-art. *IEEE Transactions on Industrial Informatics*, *15*(4), 2405-2415. <https://doi.org/10.1109/TII.2018.2873186>